# The samtools Survival Kit

**BioNexus Hub - Track 2: Genomics & NGS**

Turn a raw alignment into a sorted, indexed BAM you can actually analyze. We rebuild a small alignment from scratch so this notebook stands on its own, then practice the core samtools commands. Everything runs in Colab.

Run each cell in order with Shift+Enter.


## 1. Install the tools


In [ ]:
!apt-get -qq install -y bwa samtools
!samtools --version | head -1

## 2. Build a small alignment to work on

So this lesson is self-contained, we quickly make a tiny reference, sample reads from it, and align them - the same steps as the previous lesson, condensed.


In [ ]:
import random
random.seed(7)
bases = 'ACGT'
ref = ''.join(random.choice(bases) for _ in range(3000))
with open('reference.fasta','w') as f:
    f.write('>chr1 demo reference\n')
    for i in range(0, len(ref), 60):
        f.write(ref[i:i+60] + '\n')
L = 100
with open('reads.fastq','w') as f:
    for n in range(400):
        pos = random.randint(0, len(ref)-L)
        seq = list(ref[pos:pos+L])
        for j in range(L):
            if random.random() < 0.01:
                seq[j] = random.choice(bases)
        f.write(f'@read{n}\n{"".join(seq)}\n+\n{"I"*L}\n')
print('reference + 400 reads ready')

In [ ]:
!bwa index reference.fasta 2> /dev/null
!bwa mem reference.fasta reads.fastq 2> /dev/null | samtools view -b -o aligned.bam
print('aligned.bam created (unsorted)')

## 3. Look inside the BAM

BAM is binary, so `samtools view` is how you read it. `-H` shows just the header.


In [ ]:
!echo '--- header ---'; samtools view -H aligned.bam
!echo '--- first 3 reads (cols 1-6) ---'; samtools view aligned.bam | head -3 | cut -f1-6

## 4. Sort into genome order

The raw BAM is in sequencer order. Sorting reorders reads by position - the order every downstream tool expects.


In [ ]:
!samtools sort aligned.bam -o aligned.sorted.bam
print('sorted ->', 'aligned.sorted.bam')

## 5. Index the sorted BAM

Indexing builds a `.bai` table of contents so any region can be found instantly. You must sort first.


In [ ]:
!samtools index aligned.sorted.bam
!ls -lh aligned.sorted.bam*

## 6. Summarize, then pull a single region

`flagstat` is the health check. Then a region query returns just the reads over one spot - the payoff of indexing.


In [ ]:
!samtools flagstat aligned.sorted.bam

In [ ]:
# Reads covering positions 1000-1100 on chr1:
!samtools view aligned.sorted.bam chr1:1000-1100 | wc -l

## 7. How deep is the coverage?

Coverage (depth) is how many reads stack over each position. Higher depth means more confidence in what's really there.


In [ ]:
!samtools depth aligned.sorted.bam | head
!samtools depth aligned.sorted.bam | awk '{sum+=$3} END {print "mean depth:", sum/NR}'

## You did it

You now have a sorted, indexed BAM with a known coverage - exactly what a variant caller wants.

**Try next:** raise the read count from 400 to 2000 and watch the mean depth climb.

Next lesson: the **variant-calling capstone** - feed this BAM to a caller and produce a VCF.
